# 🚀 pgVectorDB — Quick Start

This notebook walks you through **initializing pgVectorDB**, **adding documents**, and **running your first searches**.

### Prerequisites
- PostgreSQL running with `pgvector` extension (use `docker/docker-compose.yml`)
- Python dependencies installed (`pip install -r requirements.txt`)

In [1]:
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

from langchain_core.documents import Document
from pgvectordb import pgVectorDB, Config

## 1. Initialize

We use `Config` to auto-load connection details and embedding model from `config/.env`.

In [2]:
embeddings = Config.get_embeddings()
conn_str = Config.get_connection_string()

rag = pgVectorDB(
    collection_name="nb_quickstart",
    embedding_model=embeddings,
    connection_string=conn_str,
)
await rag.initialize(overwrite_existing=True)
print("✅ Initialized")

✅ Initialized


## 2. Add Documents

Documents are standard LangChain `Document` objects with `page_content` and `metadata`.

In [3]:
docs = [
    Document(
        page_content="PostgreSQL is the world's most advanced open-source relational database.",
        metadata={"topic": "database", "year": 2024},
    ),
    Document(
        page_content="pgvector adds vector similarity search to PostgreSQL.",
        metadata={"topic": "database", "year": 2024},
    ),
    Document(
        page_content="Machine learning models convert text into dense vector embeddings.",
        metadata={"topic": "AI", "year": 2023},
    ),
    Document(
        page_content="RAG combines retrieval and generation for accurate AI responses.",
        metadata={"topic": "AI", "year": 2024},
    ),
    Document(
        page_content="HNSW indexes provide fast approximate nearest neighbor search.",
        metadata={"topic": "database", "year": 2023},
    ),
    Document(
        page_content="Docker containers package applications with all dependencies.",
        metadata={"topic": "devops", "year": 2022},
    ),
    Document(
        page_content="Kubernetes orchestrates containerized workloads at scale.",
        metadata={"topic": "devops", "year": 2023},
    ),
    Document(
        page_content="BM25 is a probabilistic ranking function used in information retrieval.",
        metadata={"topic": "AI", "year": 2022},
    ),
]

ids = await rag.add_documents(docs)
print(f"✅ Added {len(ids)} documents")
print(f"   First ID: {ids[0]}")

✅ Added 8 documents
   First ID: d35b118a-edb3-4e8d-a687-86e9807d36a0


## 3. Semantic Search

Find the most semantically similar documents to a natural language query.

In [4]:
results = await rag.semantic_search("What is vector search?", k=3)

print("🔍 Semantic Search Results:")
for i, r in enumerate(results, 1):
    print(f"  {i}. [{r['score']:.4f}] {r['content']}")
    print(f"     Metadata: {r['metadata']}")

🔍 Semantic Search Results:
  1. [0.4195] pgvector adds vector similarity search to PostgreSQL.
     Metadata: {'topic': 'database', 'year': 2024, 'langchain_id': 'af16b1ce-c870-4646-bed1-c775e63fe1cc'}
  2. [0.6500] HNSW indexes provide fast approximate nearest neighbor search.
     Metadata: {'topic': 'database', 'year': 2023, 'langchain_id': 'd2bdc45b-c65e-426b-8cb7-b1a6ee220c39'}
  3. [0.6879] BM25 is a probabilistic ranking function used in information retrieval.
     Metadata: {'topic': 'AI', 'year': 2022, 'langchain_id': 'fc01371e-3b12-43e5-9203-566989116ab7'}


## 4. Keyword Search (FTS)

Traditional full-text search using PostgreSQL's built-in `ts_rank`.

In [5]:
results = await rag.keyword_search("vector similarity", k=3)

print("🔤 Keyword (FTS) Results:")
for i, r in enumerate(results, 1):
    print(f"  {i}. [{r['score']:.4f}] {r['content']}")

🔤 Keyword (FTS) Results:
  1. [0.0608] pgvector adds vector similarity search to PostgreSQL.
  2. [0.0304] Machine learning models convert text into dense vector embeddings.


## 5. Metadata Filtering

Filter documents by JSON metadata without using any search query.

In [6]:
results = await rag.metadata_filter(filter={"topic": "AI"}, k=5)

print("📋 Metadata Filter (topic=AI):")
for i, r in enumerate(results, 1):
    print(f"  {i}. {r['content']}")
    print(f"     {r['metadata']}")

📋 Metadata Filter (topic=AI):
  1. BM25 is a probabilistic ranking function used in information retrieval.
     {'topic': 'AI', 'year': 2022, 'langchain_id': 'fc01371e-3b12-43e5-9203-566989116ab7'}
  2. Machine learning models convert text into dense vector embeddings.
     {'topic': 'AI', 'year': 2023, 'langchain_id': '7e878774-ab0c-4e6d-b454-e49bd0354733'}
  3. RAG combines retrieval and generation for accurate AI responses.
     {'topic': 'AI', 'year': 2024, 'langchain_id': 'ea705eec-25a8-4823-aa8f-7ed24dcef914'}


## 6. Metadata + Semantic Search

First filters by metadata, then ranks the filtered set by vector similarity.

In [7]:
results = await rag.metadata_semantic_search(
    query="retrieval augmented generation", filter={"topic": "AI"}, k=3
)

print("🎯 Metadata + Semantic (topic=AI):")
for i, r in enumerate(results, 1):
    print(f"  {i}. [{r['score']:.4f}] {r['content']}")

🎯 Metadata + Semantic (topic=AI):
  1. [0.4941] RAG combines retrieval and generation for accurate AI responses.
  2. [0.6307] BM25 is a probabilistic ranking function used in information retrieval.
  3. [0.7805] Machine learning models convert text into dense vector embeddings.


## 7. Hybrid Search (Semantic + Keyword)

Combines both signals using either weighted average or Reciprocal Rank Fusion (RRF).

In [8]:
# Weighted fusion
results = await rag.hybrid_search(
    query="fast database search indexing",
    k=3,
    weights=(0.6, 0.4),  # (semantic, keyword)
)

print("⚡ Hybrid Search (weighted):")
for i, r in enumerate(results, 1):
    print(f"  {i}. [{r['score']:.4f}] {r['content']}")

⚡ Hybrid Search (weighted):
  1. [1.0000] HNSW indexes provide fast approximate nearest neighbor search.
  2. [0.3754] PostgreSQL is the world's most advanced open-source relational database.
  3. [0.3593] pgvector adds vector similarity search to PostgreSQL.


In [9]:
# RRF fusion (rank-based)
results = await rag.hybrid_search(
    query="fast database search indexing",
    k=3,
    use_rrf=True,
    rrf_k=60,
)

print("⚡ Hybrid Search (RRF):")
for i, r in enumerate(results, 1):
    print(f"  {i}. [{r['score']:.4f}] {r['content']}")

⚡ Hybrid Search (RRF):
  1. [0.0328] HNSW indexes provide fast approximate nearest neighbor search.
  2. [0.0323] PostgreSQL is the world's most advanced open-source relational database.
  3. [0.0317] pgvector adds vector similarity search to PostgreSQL.


## 8. Trigram (Fuzzy) Search

Typo-tolerant substring matching using `pg_trgm`.

In [10]:
results = await rag.trigram_search(
    query="Postgreql",  # intentional typo
    k=3,
    threshold=0.1,
)

print("🔧 Trigram (Fuzzy) Results:")
for i, r in enumerate(results, 1):
    print(f"  {i}. [{r['score']:.4f}] {r['content']}")

🔧 Trigram (Fuzzy) Results:
  1. [0.1667] pgvector adds vector similarity search to PostgreSQL.
  2. [0.1111] PostgreSQL is the world's most advanced open-source relational database.


## 9. Collection Statistics

In [11]:
stats = await rag.get_stats()
print("📊 Collection Stats:")
for key, value in stats.items():
    print(f"  {key}: {value}")

📊 Collection Stats:
  index_type: hnsw
  table_name: nb_quickstart
  schema_name: public
  vector_size: 384
  index_built: False
  document_count: 8
  indexes: [{'name': 'nb_quickstart_pkey', 'definition': 'CREATE UNIQUE INDEX nb_quickstart_pkey ON public.nb_quickstart USING btree (langchain_id)'}, {'name': 'idx_nb_quickstart_content_tsvector', 'definition': 'CREATE INDEX idx_nb_quickstart_content_tsvector ON public.nb_quickstart USING gin (content_tsvector)'}, {'name': 'idx_nb_quickstart_content_trgm', 'definition': 'CREATE INDEX idx_nb_quickstart_content_trgm ON public.nb_quickstart USING gin (content gin_trgm_ops)'}]
  table_size: 120 kB


## 10. Cleanup

In [12]:
await rag.delete_table()
await rag.close()
print("🧹 Cleaned up")

🧹 Cleaned up
